In [28]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier, Pool
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# ======================================================================
# CONFIGURAR JUPYTER PARA NÃO TRUNCAR OUTPUT
# ======================================================================
pd.set_option('display.max_rows', 999)
pd.set_option('display.max_columns', 999)
pd.set_option('display.width', 300)
pd.set_option('display.max_colwidth', None)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

SEPARATOR = "=" * 100


# ======================================================================
# LOAD DATA
# ======================================================================
print(f"\n{SEPARATOR}\n=== LOAD DATA ===\n{SEPARATOR}")
train = pd.read_csv("datasets/training_data.csv", encoding="latin1")
test  = pd.read_csv("datasets/test_data.csv", encoding="latin1")
example = pd.read_csv("datasets/example_submission.csv", encoding="latin1")

print("Train:", train.shape)
print("Test:",  test.shape)

# ======================================================================
# TARGET
# ======================================================================
print(f"\n{SEPARATOR}\n=== PREPARAR TARGET ===\n{SEPARATOR}")
y = train["AVERAGE_SPEED_DIFF"].fillna("None")
X = train.drop("AVERAGE_SPEED_DIFF", axis=1)


# ======================================================================
# PROCESS DATES
# ======================================================================
print(f"\n{SEPARATOR}\n=== PROCESSAR DATAS ===\n{SEPARATOR}")

def process_dates(df):
    df["record_date"] = pd.to_datetime(df["record_date"])
    df["year"] = df["record_date"].dt.year
    df["month"] = df["record_date"].dt.month
    df["day"] = df["record_date"].dt.day
    df["hour"] = df["record_date"].dt.hour
    df["weekday"] = df["record_date"].dt.weekday
    return df.drop("record_date", axis=1)

X = process_dates(X)
test = process_dates(test)

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(include=["number"]).columns.tolist()

print("CATEG:", cat_cols)
print("NUM:", num_cols)


# ======================================================================
# NAN FIX
# ======================================================================
print(f"\n{SEPARATOR}\n=== FIX NAN ===\n{SEPARATOR}")

for col in cat_cols:
    X[col] = X[col].fillna("Unknown")

for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

print("NAN resolvidos.")


# ======================================================================
# TRAIN / VAL SPLIT
# ======================================================================
print(f"\n{SEPARATOR}\n=== SPLIT ===\n{SEPARATOR}")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train split:", X_train.shape)
print("Val split:",   X_val.shape)


# ======================================================================
# CATBOOST
# ======================================================================
print(f"\n{SEPARATOR}\n=== TREINAR CATBOOST ===\n{SEPARATOR}")

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
val_pool   = Pool(X_val, cat_features=cat_cols)

cat_model = CatBoostClassifier(
    iterations=700,
    depth=8,
    learning_rate=0.04,
    loss_function="MultiClass",
    verbose=False     # <<< SILENCIADO PARA VER RESTO DOS MODELOS
)

cat_model.fit(train_pool, eval_set=val_pool, verbose=False)

cat_preds = cat_model.predict(val_pool).flatten()

print("\nCATBOOST ACC:", accuracy_score(y_val, cat_preds))
print(classification_report(y_val, cat_preds))


# ======================================================================
# LIGHTGBM + RF → REQUIREM ENCODING
# ======================================================================
print(f"\n{SEPARATOR}\n=== ENCODING PARA LGBM / RF ===\n{SEPARATOR}")

# Criar encoding seguro
encoders = {}
Xl_train = X_train.copy()
Xl_val = X_val.copy()

for col in cat_cols:
    le = LabelEncoder()
    Xl_train[col] = le.fit_transform(Xl_train[col].astype(str))
    Xl_val[col]   = X_val[col].map(
        lambda x: x if x in le.classes_ else le.classes_[0]
    ).astype(str)
    Xl_val[col]   = le.transform(Xl_val[col])
    encoders[col] = le

target_enc = LabelEncoder()
y_train_enc = target_enc.fit_transform(y_train)
y_val_enc   = target_enc.transform(y_val)


# ======================================================================
# LIGHTGBM
# ======================================================================
print(f"\n{SEPARATOR}\n=== TREINAR LIGHTGBM ===\n{SEPARATOR}")

lgbm = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.04,
    num_leaves=60,
    objective="multiclass",
    num_class=len(target_enc.classes_)
)

lgbm.fit(Xl_train, y_train_enc)
lgb_preds_enc = lgbm.predict(Xl_val)
lgb_preds = target_enc.inverse_transform(lgb_preds_enc)

print("\nLIGHTGBM ACC:", accuracy_score(y_val, lgb_preds))
print(classification_report(y_val, lgb_preds))


# ======================================================================
# RANDOM FOREST
# ======================================================================
print(f"\n{SEPARATOR}\n=== TREINAR RANDOM FOREST ===\n{SEPARATOR}")

rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=18,
    n_jobs=-1
)

rf.fit(Xl_train, y_train_enc)
rf_preds_enc = rf.predict(Xl_val)
rf_preds = target_enc.inverse_transform(rf_preds_enc)

print("\nRANDOM FOREST ACC:", accuracy_score(y_val, rf_preds))
print(classification_report(y_val, rf_preds))


# ======================================================================
# STACKING
# ======================================================================
print(f"\n{SEPARATOR}\n=== STACKING ===\n{SEPARATOR}")

cat_p = cat_model.predict_proba(val_pool)
lgb_p = lgbm.predict_proba(Xl_val)
rf_p  = rf.predict_proba(Xl_val)

stack_input = np.hstack([cat_p, lgb_p, rf_p])

meta = LogisticRegression(max_iter=2000)
meta.fit(stack_input, y_val)

stack_preds = meta.predict(stack_input)

print("\nSTACKING ACC:", accuracy_score(y_val, stack_preds))
print(classification_report(y_val, stack_preds))


# ======================================================================
# SUBMISSION
# ======================================================================
print(f"\n{SEPARATOR}\n=== SUBMISSION ===\n{SEPARATOR}")

full = pd.concat([X, test], ignore_index=True)

# Fix NAN test
for col in cat_cols:
    full[col] = full[col].fillna("Unknown")
for col in num_cols:
    full[col] = full[col].fillna(full[col].median())

Xc_test = full.iloc[len(train):]
Xl_test = Xc_test.copy()

# Encode test
for col in cat_cols:
    le = encoders[col]
    Xl_test[col] = Xc_test[col].map(lambda x: x if x in le.classes_ else le.classes_[0])
    Xl_test[col] = le.transform(Xl_test[col].astype(str))

test_pool = Pool(Xc_test, cat_features=cat_cols)

cat_p = cat_model.predict_proba(test_pool)
lgb_p = lgbm.predict_proba(Xl_test)
rf_p  = rf.predict_proba(Xl_test)

stack_final = np.hstack([cat_p, lgb_p, rf_p])
final_preds = meta.predict(stack_final)

submission = pd.DataFrame({
    "RowId": example["RowId"],
    "Speed_Diff": final_preds
})

submission.to_csv("submission.csv", index=False)
print("✔ submission.csv CREATED!")




=== LOAD DATA ===
Train: (6812, 14)
Test: (1500, 13)

=== PREPARAR TARGET ===

=== PROCESSAR DATAS ===
CATEG: ['city_name', 'LUMINOSITY', 'AVERAGE_CLOUDINESS', 'AVERAGE_RAIN']
NUM: ['AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF', 'AVERAGE_FREE_FLOW_TIME', 'AVERAGE_TEMPERATURE', 'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY', 'AVERAGE_WIND_SPEED', 'AVERAGE_PRECIPITATION', 'year', 'month', 'day', 'hour', 'weekday']

=== FIX NAN ===
NAN resolvidos.

=== SPLIT ===
Train split: (5449, 17)
Val split: (1363, 17)

=== TREINAR CATBOOST ===



CATBOOST ACC: 0.8070432868672047
              precision    recall  f1-score   support

        High       0.76      0.73      0.75       213
         Low       0.73      0.73      0.73       284
      Medium       0.78      0.78      0.78       330
        None       0.88      0.92      0.90       440
   Very_High       0.86      0.82      0.84        96

    accuracy                           0.81      1363
   macro avg       0.80      0.79      0.80      1363
weighted avg       0.81      0.81      0.81      1363


=== ENCODING PARA LGBM / RF ===

=== TREINAR LIGHTGBM ===
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000458 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 977
[LightGBM] [Info] Number of data points in the train set: 5449, number of used features: 15
[LightGBM] [Info] Start training from score -1.857951
[LightGBM] [Info] Start training from score -1.568799
[LightGBM] [Info] Start